# 模块 5 节点壳（`/visual_tracker`）—— 消息互转

演示 `scripts/visual_tracker_node.py` 中**不依赖 ROS master 的纯函数**：

- `detections_from_msg(msg)` —— `uav_tracking/Detections` → `list[DetectedTarget]`
- `visual_tracks_to_msg(tracks)` —— `list[VisualTrack]` → `uav_tracking/VisualTrackArray`

> **运行前提**：已 `source /opt/ros/noetic/setup.bash` 与
> `source catkin_ws/devel/setup.bash`。

## 0. 环境准备

In [ ]:
import os, sys

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))   # ai4/
SCRIPTS = os.path.join(REPO, "catkin_ws", "src", "uav_tracking", "scripts")
LIB = os.path.join(REPO, "catkin_ws", "src", "uav_tracking", "lib")
for p in (os.path.join(REPO, "code"), SCRIPTS, LIB):
    if p not in sys.path:
        sys.path.insert(0, p)
print("REPO =", REPO)

try:
    import rospy
    print("rospy OK")
except ImportError as exc:
    print("[阻塞] 未能 import rospy，请先 source ROS 环境：", exc)

## 1. `detections_from_msg` —— 消息 → 纯数据

注意：`uav_tracking.msg.VisualTrack` 与 `datatypes.VisualTrack` **同名但不同物**
（前者是 ROS 消息、后者是纯数据结构），转换函数负责二者互转。
这也是 `state_machine_node.py` 注释里说的"起别名模块避免冲突"。

In [ ]:
try:
    import visual_tracker_node as vtn
    from datatypes import DetectedTarget as DetectedTargetData
    from uav_tracking.msg import DetectedTarget as DetectedTargetMsg
    from uav_tracking.msg import Detections

    # --- 空消息 ---
    print("空消息 ->", vtn.detections_from_msg(Detections()))

    # --- 有目标 ---
    msg = Detections()
    t = DetectedTargetMsg(); t.class_id = 2; t.confidence = 0.77; t.bbox = [10.0, 20.0, 30.0, 40.0]
    msg.targets.append(t)
    out = vtn.detections_from_msg(msg)
    d = out[0]
    print("转换结果 -> 类型:", type(d).__name__, "（纯逻辑层，非 ROS 消息）")
    print("            class_id=%d confidence=%.2f bbox=%s" % (d.class_id, d.confidence, d.bbox))
    print("            isinstance(DetectedTarget(纯数据)) =", isinstance(d, DetectedTargetData))
except ImportError as exc:
    print("[跳过] 需先 source ROS 并 catkin_make：", exc)

## 2. `visual_tracks_to_msg` —— 含「空也发」

**关键契约**：无航迹时必须产出**合法**的空 `VisualTrackArray`（`tracks` 为空数组）。

In [ ]:
try:
    from datatypes import VisualTrack as VisualTrackData

    empty = vtn.visual_tracks_to_msg([])
    print("空列表 ->", type(empty).__name__, " tracks =", len(empty.tracks))

    msg = vtn.visual_tracks_to_msg([
        VisualTrackData(id=7, bbox=(280.0, 200.0, 120.0, 90.0), area_ratio=0.0352),
    ])
    it = msg.tracks[0]
    print("有航迹 -> tracks =", len(msg.tracks))
    print("            id=%d bbox=%s area_ratio=%.4f"
          % (it.id, [round(float(v), 1) for v in it.bbox], it.area_ratio))
except ImportError as exc:
    print("[跳过] 需先 source ROS 并 catkin_make：", exc)

## 3. 端到端（消息 → 跟踪 → 消息）

把两个转换函数与纯逻辑跟踪器串起来，等价于节点 `on_detections` 回调的内部流程。

In [ ]:
try:
    import visual_tracker as vt
    from uav_tracking.msg import Detections as DetectionsMsg
    from uav_tracking.msg import DetectedTarget as DetectedTargetMsg2

    vt.reset()
    vt.setup(frame_size=(640, 480))

    # 构造两帧检测消息：目标移动，ID 应保持
    ids = []
    for i in range(4):
        m = DetectionsMsg()
        tt = DetectedTargetMsg2()
        tt.class_id = 0; tt.confidence = 0.9
        tt.bbox = [280.0 + i * 10.0, 200.0 + i * 6.0, 64.0, 48.0]
        m.targets.append(tt)
        tracks = vt.track(vtn.detections_from_msg(m))
        out_msg = vtn.visual_tracks_to_msg(tracks)
        ids.append(int(out_msg.tracks[0].id) if out_msg.tracks else None)

    print("4 帧的 ID:", ids, " -> ID 稳定:", len(set(ids)) == 1)

    # 空输入的「空也发」
    tracks_empty = vt.track(vtn.detections_from_msg(DetectionsMsg()))
    print("空输入 -> 航迹数:", len(tracks_empty),
          " 消息 tracks 长度:", len(vtn.visual_tracks_to_msg(tracks_empty).tracks))
    vt.reset()
except ImportError as exc:
    print("[跳过] 需先 source ROS 并 catkin_make：", exc)

## 4. 端到端 E2E

```bash
bash test/visual_tracker/e2e_test/run_e2e.sh
```

- **阶段 A（有目标）**：断言航迹产出、`area_ratio` 与 `bbox/(画面面积)` 一致、
  **ID 跨帧稳定**；
- **阶段 B（无目标）**：断言话题仍在流且 `tracks` 恒为空（「空也发」契约）。